# AgroClima RS — Previsão de Chuva D+1 no Sul do Rio Grande do Sul
## Região Geográfica Intermediária de Pelotas (IBGE 4302)

Este notebook realiza a análise exploratória e a modelagem preditiva para prever a ocorrência de chuva no dia seguinte (D+1) a partir de medições meteorológicas diárias do INMET.

### Estrutura do trabalho
1. Carregamento e diagnóstico dos dados meteorológicos (10 estações, ano de 2026).
2. Análise exploratória das variáveis físicas (pressão, umidade, radiação).
3. Divisão temporal estrita (treino até junho, teste em julho e agosto).
4. Treinamento e comparação de modelos (Regressão Logística, Random Forest e XGBoost).
5. Explicabilidade e estrutura para conexão futura com dados agrícolas oficiais (IBGE/CONAB).

*Instruções para o Colab:* Faça o upload do arquivo `features_rain_d1_sul_rs_2026.csv` antes de executar.


## 1. Configuração do Ambiente


Instalação de pacotes e definição de parâmetros básicos.


In [ ]:
!pip install -q xgboost scikit-learn seaborn matplotlib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelagem e Pré-processamento
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10
np.random.seed(42)

print("✅ Ambiente configurado com sucesso!")


## 2. Carga dos Dados (`features_rain_d1_sul_rs_2026.csv`)


In [ ]:
CSV_NAME = "features_rain_d1_sul_rs_2026.csv"
candidates = [CSV_NAME, f"/content/{CSV_NAME}", os.path.expanduser(f"~/Downloads/{CSV_NAME}")]

csv_path = None
for p in candidates:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    print(f"⚠️ Arquivo '{CSV_NAME}' não encontrado. Faça o upload abaixo:")
    from google.colab import files
    uploaded = files.upload()
    csv_path = list(uploaded.keys())[0]

# Carregamento com separador ';' e decimal ','
df = pd.read_csv(csv_path, sep=';', decimal=',')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['station_id', 'date']).reset_index(drop=True)

print(f"✅ Base carregada: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"📅 Período: {df['date'].min().strftime('%d/%m/%Y')} até {df['date'].max().strftime('%d/%m/%Y')}")
print(f"🌧️ Taxa de Ocorrência do Alvo ('rain_tomorrow'): {df['rain_tomorrow'].mean():.2%}")
df.head()


## 3. Dicionário de Dados e Diagnóstico

A chave primária é composta por `station_id` e `date`. Cada linha representa a observação diária consolidada de uma estação meteorológica específica.


In [ ]:
# Diagnóstico geral da base carregada
quality_summary = pd.DataFrame({
    'indicador': [
        'linhas', 'colunas', 'estações', 'municípios', 'data inicial', 'data final',
        'duplicidades station_id + date', 'valores ausentes totais'
    ],
    'valor': [
        len(df), df.shape[1], df['station_id'].nunique(), df['codigo_ibge'].nunique(),
        df['date'].min().strftime('%d/%m/%Y'), df['date'].max().strftime('%d/%m/%Y'),
        df.duplicated(['station_id', 'date']).sum(), int(df.isna().sum().sum())
    ]
})
display(quality_summary)

missing = (df.isna().sum().rename('ausentes').to_frame()
           .assign(percentual=lambda x: (x['ausentes'] / len(df) * 100).round(2))
           .query('ausentes > 0').sort_values('ausentes', ascending=False))
print('Variáveis com valores ausentes:')
display(missing)

print('Cobertura por estação:')
coverage = df.groupby(['station_id', 'municipio']).agg(
    inicio=('date','min'), fim=('date','max'), registros=('date','size')
).reset_index()
display(coverage)


## 4. Análise Exploratória

Comportamento da precipitação acumulada e relação de variáveis físicas com a chuva no dia seguinte.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Total de chuva acumulada por município
precip_mun = df.groupby('municipio')['precip_mm'].sum().sort_values()
precip_mun.plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title("Precipitação Total Acumulada no Período (mm)")
axes[0, 0].set_xlabel("Chuva Acumulada (mm)")

# 2. Queda de Pressão Barométrica 24h vs Chuva Amanhã
sns.kdeplot(data=df, x='pressure_change_24h', hue='rain_tomorrow', common_norm=False, ax=axes[0, 1], palette=['navy', 'crimson'])
axes[0, 1].set_title("Queda de Pressão (hPa/24h) vs Chuva Amanhã (D+1)")
axes[0, 1].set_xlabel("Delta Pressão 24h (hPa)")

# 3. Umidade Relativa vs Chuva Amanhã
sns.boxplot(data=df, x='rain_tomorrow', y='humidity_avg', ax=axes[1, 0], palette=['lightblue', 'lightcoral'])
axes[1, 0].set_title("Umidade Relativa Hoje vs Chuva Amanhã")
axes[1, 0].set_xticklabels(["Não Chove (0)", "Chove (1)"])

# 4. Radiação Solar vs Temperatura Máxima
sns.scatterplot(data=df, x='solar_radiation', y='temp_max', hue='rain_tomorrow', alpha=0.6, ax=axes[1, 1], palette=['blue', 'orange'])
axes[1, 1].set_title("Radiação Solar vs Temperatura Máxima")
axes[1, 1].set_xlabel("Radiação Solar (MJ/m²)")

plt.tight_layout()
plt.show()


## 5. Modelagem com Validação Temporal

A divisão temporal utiliza dados de janeiro a junho para treino (1.685 registros) e julho a agosto para teste (590 registros), simulando um cenário operacional real.


In [ ]:
FEATURE_COLS = [
    'precip_mm', 'precip_lag_1d', 'precip_lag_2d',
    'precip_sum_3d', 'precip_sum_7d', 'rain_days_7d', 'dry_days',
    'temp_min', 'temp_max', 'temp_avg', 'temp_avg_3d',
    'humidity_avg', 'humidity_avg_3d',
    'pressure_avg', 'pressure_change_24h',
    'wind_speed', 'solar_radiation',
    'day_of_year_sin', 'day_of_year_cos',
    'altitude', 'lat', 'lon'
]

TARGET = 'rain_tomorrow'

split_date = '2026-07-01'
train_mask = df['date'] < split_date
test_mask = df['date'] >= split_date

X_train, y_train = df.loc[train_mask, FEATURE_COLS], df.loc[train_mask, TARGET]
X_test, y_test = df.loc[test_mask, FEATURE_COLS], df.loc[test_mask, TARGET]

# Imputação ajustada apenas no treino para evitar vazamento de informação.
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURE_COLS, index=X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns=FEATURE_COLS, index=X_test.index)

print(f"📊 Amostras Treino (Jan-Jun/2026): {len(X_train):,}")
print(f"📊 Amostras Teste  (Jul-Ago/2026): {len(X_test):,}")

# 1. Regressão Logística (com scaler)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
prob_lr = lr.predict_proba(X_test_s)[:, 1]
pred_lr = (prob_lr >= 0.30).astype(int)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=4, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
prob_rf = rf.predict_proba(X_test)[:, 1]
pred_rf = (prob_rf >= 0.30).astype(int)

# 3. XGBoost
xgb = XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.04, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
prob_xgb = xgb.predict_proba(X_test)[:, 1]
pred_xgb = (prob_xgb >= 0.30).astype(int)

print("✅ Modelos treinados com sucesso!")


## 6. Avaliação dos Modelos

Comparativo de desempenho na partição de teste (ROC-AUC e F1-Score com threshold em 0.30).


In [ ]:
results = [
    {"Modelo": "Logistic Regression", "ROC-AUC": round(roc_auc_score(y_test, prob_lr), 4), "F1-Score": round(f1_score(y_test, pred_lr), 4)},
    {"Modelo": "Random Forest", "ROC-AUC": round(roc_auc_score(y_test, prob_rf), 4), "F1-Score": round(f1_score(y_test, pred_rf), 4)},
    {"Modelo": "XGBoost", "ROC-AUC": round(roc_auc_score(y_test, prob_xgb), 4), "F1-Score": round(f1_score(y_test, pred_xgb), 4)},
]

df_results = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
display(df_results)

print("\n--- Relatório Detalhado de Classificação (XGBoost) ---")
print(classification_report(y_test, pred_xgb, target_names=["Sem Chuva", "Chuva"]))


## 7. Importância das Variáveis

Relevância das variáveis preditivas no modelo XGBoost.


In [ ]:
importances = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.tail(15).plot(kind='barh', color='teal')
plt.title("Top 15 Variáveis Mais Relevantes para Prever Chuva D+1 no Sul do RS")
plt.xlabel("Importância Relativa (F-Score)")
plt.tight_layout()
plt.show()


## 8. Integração com Dados Agrícolas (Fase 2)

Espaço reservado para cruzamento com bases oficiais da PAM/IBGE ou CONAB via `codigo_ibge` e `year`.


In [ ]:
# Não gerar dados agrícolas artificiais. Faça upload de uma tabela real antes de executar.
# Formato mínimo esperado: codigo_ibge, municipio, year, crop,
# area_planted_ha, area_harvested_ha, production_t, yield_kg_ha.
AGRI_NAME = 'agriculture_pam_conab.csv'
agri_candidates = [AGRI_NAME, f'/content/{AGRI_NAME}']
agri_path = next((p for p in agri_candidates if os.path.exists(p)), None)

if agri_path is None:
    print('ℹ️ Fase agrícola aguardando arquivo real da PAM/IBGE ou CONAB.')
    print('Envie um CSV com chave codigo_ibge + year e as variáveis agrícolas documentadas.')
else:
    df_agri = pd.read_csv(agri_path, sep=None, engine='python', decimal=',')
    required = {'codigo_ibge', 'year', 'crop', 'production_t', 'yield_kg_ha'}
    missing_required = required - set(df_agri.columns)
    if missing_required:
        raise ValueError(f'Colunas ausentes na tabela agrícola: {sorted(missing_required)}')
    clim_summary = df.groupby('codigo_ibge').agg(
        precip_total_mm=('precip_mm', 'sum'),
        rain_days=('precip_mm', lambda x: (x >= 1.0).sum()),
        temp_avg=('temp_avg', 'mean'),
        solar_rad_avg=('solar_radiation', 'mean')
    ).reset_index()
    df_merged = pd.merge(df_agri, clim_summary, on='codigo_ibge', how='inner')
    print('Tabela integrada com dados agrícolas reais:')
    display(df_merged.head())


## 9. Limitações e Trabalhos Futuros

- **Janela temporal:** A base atual cobre 8 meses de 2026. Para uso agronômico prático, é necessário ingerir histórico de múltiplos anos.
- **Dados agrícolas:** A análise de produtividade depende do fechamento anual das safras pelo IBGE.
- **Deploy:** O pipeline de ingestão pode ser agendado em rotina diária na nuvem (AWS Lambda + S3).
